# YTAgent — Google Colab Free GPU Worker

Alternative to Kaggle using Colab's free GPU access. GPU availability and session duration are not guaranteed. This runs the same Wan 2.1 + Kokoro worker. Keep the tab/session active during generation. No paid Colab, paid APIs, or paid GPU fallback is used.

**Important:** Colab free resources are dynamic and intended for interactive notebook use. Do not use this notebook to bypass usage restrictions or expect unattended always-on compute.

In [ ]:
import os, subprocess
repo_dir = '/content/YTAgent'
if not os.path.isdir(repo_dir):
    subprocess.run(['git','clone','https://github.com/qingremzy/YTAgent.git',repo_dir],check=True)
else:
    subprocess.run(['git','-C',repo_dir,'pull','--ff-only','origin','main'],check=True)
os.chdir(repo_dir + '/gpu-worker')
print('Installing worker dependencies...')
subprocess.run(['python','-m','pip','install','-r','requirements.txt'],check=True)
print('Installing Kokoro and Misaki without incompatible optional extras...')
subprocess.run(['python','-m','pip','install','--no-deps','git+https://github.com/hexgrad/misaki.git'],check=True)
subprocess.run(['python','-m','pip','install','--no-deps','git+https://github.com/hexgrad/kokoro.git'],check=True)
subprocess.run(['apt-get','-qq','-y','update'],check=True)
subprocess.run(['apt-get','-qq','-y','install','espeak-ng'],check=True)
print('Setup completed.')

In [ ]:
import torch
print('CUDA available:',torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No free GPU was assigned. In Runtime > Change runtime type, select GPU if available, then reconnect.')
print('GPU:',torch.cuda.get_device_name(0))
from google.colab import userdata
token = userdata.get('YT_AGENT_WORKER_TOKEN')
if not token:
    raise RuntimeError('Add YT_AGENT_WORKER_TOKEN in Colab Secrets (key icon). Keep it private and enable notebook access.')
os.environ['YT_AGENT_WORKER_TOKEN'] = token
os.environ['YT_AGENT_SERVER_URL'] = 'https://ytagent-production-1038.up.railway.app'
print('Private worker token loaded. It will not be displayed.')

In [ ]:
import subprocess,sys
print('Waiting for one queued YTAgent job. This can take a long time on a free GPU.')
subprocess.run([sys.executable,'app.py','--run-once','--wait-seconds','1200'],check=True)
print('Worker finished. Check YTAgent for the finished video.')